# Agent 3: Response Generator — Evaluation

Evaluate the fine-tuned LLaMA-3.2-3B-Instruct QLoRA adapter on test data.

**Metrics:**
- Format compliance (both Confirmation + Instructions present)
- Confirmation present rate
- Instructions present rate
- BLEU score (generated vs reference)
- Average response length
- Per-department & per-urgency breakdown

In [ ]:
!pip install -q transformers>=4.45.0 peft>=0.13.0 bitsandbytes>=0.44.0 accelerate>=1.0.0 datasets huggingface_hub nltk

In [ ]:
import shutil, os
from google.colab import drive

if not os.path.ismount('/content/drive'):
    if os.path.exists('/content/drive') and os.listdir('/content/drive'):
        shutil.rmtree('/content/drive')
    drive.mount('/content/drive')
else:
    print('Drive already mounted.')

In [ ]:
from huggingface_hub import login

try:
    from google.colab import userdata
    hf_token = userdata.get('HF_TOKEN')
    login(token=hf_token)
    print('Logged in to HuggingFace via Colab Secrets')
except Exception:
    hf_token = os.environ.get('HF_TOKEN', '')
    if hf_token:
        login(token=hf_token)
        print('Logged in via env var')
    else:
        print('WARNING: No HF_TOKEN found!')

In [ ]:
import torch

# ============================================================
# CONFIG
# ============================================================
MODEL_ID = 'meta-llama/Llama-3.2-3B-Instruct'
ADAPTER_DIR = '/content/drive/MyDrive/response_generator/final_adapter'
TEST_FILE = '/content/drive/MyDrive/response_generator_test.jsonl'
EVAL_LIMIT = 200  # Set to None for full eval (2931 samples)

if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)} | VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')
else:
    raise RuntimeError('No GPU detected!')

In [ ]:
import json, random

with open(TEST_FILE) as f:
    test_data = [json.loads(line) for line in f]

if EVAL_LIMIT:
    random.seed(42)
    test_data = random.sample(test_data, min(EVAL_LIMIT, len(test_data)))

print(f'Loaded {len(test_data)} test samples')
print(f'Sample keys: {list(test_data[0].keys())}')
print(f'Sample input: {test_data[0]["input"][:200]}...')

In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from peft import PeftModel

tokenizer = AutoTokenizer.from_pretrained(ADAPTER_DIR)

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type='nf4',
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_use_double_quant=True,
)

base_model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    quantization_config=bnb_config,
    device_map='auto',
)

model = PeftModel.from_pretrained(base_model, ADAPTER_DIR)
model.eval()
print('Model loaded successfully.')

In [ ]:
import sys
import nltk
from collections import defaultdict
from nltk.translate.bleu_score import sentence_bleu, SmoothingFunction

nltk.download('punkt', quiet=True)
nltk.download('punkt_tab', quiet=True)
smooth = SmoothingFunction().method1

SYSTEM_PROMPT = (
    'You are a compassionate medical assistant. A patient has been assigned '
    'an appointment. Write a warm, clear appointment confirmation and practical '
    'pre-visit instructions. Keep the tone professional but reassuring. '
    'Format your response as:\n'
    'Confirmation: <one sentence confirming the appointment>\n'
    'Instructions: <2-4 specific pre-visit instructions>'
)

has_conf, has_inst, format_ok = 0, 0, 0
total_length = 0
bleu_scores = []

dept_counts = defaultdict(lambda: {'total': 0, 'format_ok': 0, 'bleu_sum': 0.0})
urg_counts = defaultdict(lambda: {'total': 0, 'format_ok': 0, 'bleu_sum': 0.0})

sample_outputs = []
total = len(test_data)

for i, sample in enumerate(test_data):
    messages = [
        {'role': 'system', 'content': SYSTEM_PROMPT},
        {'role': 'user', 'content': sample['input']},
    ]
    tokenized = tokenizer.apply_chat_template(
        messages, add_generation_prompt=True, return_tensors='pt', return_dict=True
    ).to(model.device)
    input_len = tokenized['input_ids'].shape[-1]

    with torch.inference_mode():
        outputs = model.generate(
            **tokenized, max_new_tokens=512, temperature=0.7,
            do_sample=True, top_p=0.9, pad_token_id=tokenizer.eos_token_id,
        )

    generated = tokenizer.decode(outputs[0][input_len:], skip_special_tokens=True).strip()
    total_length += len(generated)

    # Format compliance
    has_c = 'Confirmation:' in generated or 'confirmed' in generated.lower()
    has_i = 'Instructions:' in generated or 'instruction' in generated.lower()

    if has_c: has_conf += 1
    if has_i: has_inst += 1
    if has_c and has_i: format_ok += 1

    # BLEU score vs reference
    ref_tokens = nltk.word_tokenize(sample['output'].lower())
    gen_tokens = nltk.word_tokenize(generated.lower())
    bleu = sentence_bleu([ref_tokens], gen_tokens, smoothing_function=smooth)
    bleu_scores.append(bleu)

    dept = sample.get('department', 'Unknown')
    urg = sample.get('urgency', 'Unknown')
    dept_counts[dept]['total'] += 1
    urg_counts[urg]['total'] += 1
    dept_counts[dept]['bleu_sum'] += bleu
    urg_counts[urg]['bleu_sum'] += bleu
    if has_c and has_i:
        dept_counts[dept]['format_ok'] += 1
        urg_counts[urg]['format_ok'] += 1

    if len(sample_outputs) < 5:
        sample_outputs.append({
            'input': sample['input'][:150],
            'reference': sample['output'][:200],
            'generated': generated[:300],
            'bleu': bleu,
        })

    done = i + 1
    pct = done / total
    bar = chr(9608) * int(pct * 30) + chr(9617) * (30 - int(pct * 30))
    avg_bleu = sum(bleu_scores) / len(bleu_scores)
    comp = format_ok / done
    print(f'\r  {bar} {done}/{total} ({pct:.0%}) | compliance: {comp:.1%} | BLEU: {avg_bleu:.3f}', end='')
    sys.stdout.flush()

print(f'\n\nEvaluation complete. Processed {total} samples.')

In [ ]:
import numpy as np

n = len(test_data)
avg_bleu = np.mean(bleu_scores)
median_bleu = np.median(bleu_scores)

print('=' * 60)
print('RESPONSE GENERATOR EVALUATION REPORT')
print('=' * 60)
print(f'Total samples:         {n}')
print(f'Format compliance:     {format_ok/n:.1%}')
print(f'Confirmation present:  {has_conf/n:.1%}')
print(f'Instructions present:  {has_inst/n:.1%}')
print(f'Avg response length:   {total_length/n:.0f} chars')
print(f'BLEU (mean):           {avg_bleu:.4f}')
print(f'BLEU (median):         {median_bleu:.4f}')

# Per-Department
print(f'\n--- Per-Department ---')
print(f'{"Department":<22} {"Compliance":>10} {"Avg BLEU":>10} {"Samples":>8}')
for dept, c in sorted(dept_counts.items()):
    comp = c['format_ok'] / c['total'] if c['total'] > 0 else 0.0
    dept_bleu = c['bleu_sum'] / c['total'] if c['total'] > 0 else 0.0
    print(f'{dept:<22} {comp:>10.1%} {dept_bleu:>10.4f} {c["total"]:>8}')

# Per-Urgency
print(f'\n--- Per-Urgency ---')
print(f'{"Urgency":<22} {"Compliance":>10} {"Avg BLEU":>10} {"Samples":>8}')
for urg, c in sorted(urg_counts.items()):
    comp = c['format_ok'] / c['total'] if c['total'] > 0 else 0.0
    urg_bleu = c['bleu_sum'] / c['total'] if c['total'] > 0 else 0.0
    print(f'{urg:<22} {comp:>10.1%} {urg_bleu:>10.4f} {c["total"]:>8}')

In [ ]:
# Show sample outputs for manual review
print('\n--- Sample Outputs (with Reference Comparison) ---')
for i, s in enumerate(sample_outputs):
    print(f'\n[Sample {i+1}] BLEU: {s["bleu"]:.4f}')
    print(f'Input:     {s["input"]}...')
    print(f'Reference: {s["reference"]}...')
    print(f'Generated: {s["generated"]}...')
    print('-' * 50)

In [ ]:
# Save results to Drive
results = {
    'total': n,
    'format_compliance': format_ok / n,
    'confirmation_rate': has_conf / n,
    'instructions_rate': has_inst / n,
    'avg_response_length': total_length / n,
    'bleu_mean': float(avg_bleu),
    'bleu_median': float(median_bleu),
}

output_path = '/content/drive/MyDrive/eval_response_generator_results.json'
with open(output_path, 'w') as f:
    json.dump(results, f, indent=2)
print(f'Results saved to {output_path}')